In [1]:
import csv, hashlib, json, platform
from importlib.metadata import version
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd

In [2]:
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "allstate_claims_data.csv").is_file())
SOURCE = ROOT / "data" / "allstate_claims_data.csv"
GATE2 = ROOT / "artifacts" / "october" / "gate 2"
OFFICIAL = ROOT / "artifacts" / "october" / "gate 3" / "comparison_inputs" / "glm-20261103T055706Z-0748a4b5"
OUT = ROOT / "artifacts" / "october" / "gate 3" / "runs" / "glm-repro-20261110-rachel"

def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

config = json.loads((GATE2 / "approach1_glm_config.json").read_text())
official = json.loads((OFFICIAL / "run_receipt.json").read_text())
assert sha256(GATE2 / "approach1_glm_config.json") == official["input_sha256"]["artifacts/october/gate 2/approach1_glm_config.json"]

manifest = pd.read_csv(ROOT / "artifacts" / "october" / "gate 1" / "october_partition_manifest.csv")
train_ids = set(manifest.loc[manifest.partition.eq("train"), "id"])
valid_ids = set(manifest.loc[manifest.partition.eq("validation"), "id"])
folds = pd.read_csv(GATE2 / "cv_folds" / "october_cv_folds.csv").set_index("id").sort_index()

with SOURCE.open(newline="") as f:
    y = pd.Series({int(r["id"]): float(r["loss"]) for r in csv.DictReader(f) if int(r["id"]) in train_ids}).sort_index()

CATS = [f"cat{i}" for i in range(1, 117)]
CONTS = [f"cont{i}" for i in range(1, 15)]
FEATURES = CATS + CONTS
types = {"id": "int64", **{c: "string" for c in CATS}, **{c: "float64" for c in CONTS}}
X = pd.concat(chunk[chunk.id.isin(train_ids | valid_ids)] for chunk in
              pd.read_csv(SOURCE, usecols=["id"] + FEATURES, dtype=types, chunksize=20_000)).set_index("id")
X_train, X_valid = X.loc[y.index, FEATURES], X.loc[sorted(valid_ids), FEATURES]

assert set(y.index) == train_ids and set(folds.index) == train_ids
print("Training:", X_train.shape, "| Validation inputs:", X_valid.shape)
print({p: version(p) for p in ["pandas", "numpy", "scipy", "scikit-learn"]})

Training: (141238, 130) | Validation inputs: (28248, 130)
{'pandas': '2.3.2', 'numpy': '2.3.3', 'scipy': '1.18.1', 'scikit-learn': '1.7.2'}


In [3]:
gate2_nb = json.loads((ROOT / "notebooks" / "gate2_training_folds.ipynb").read_text())
for cell in gate2_nb["cells"]:
    source = "".join(cell["source"])
    if cell["cell_type"] == "code" and ("class LogTargetRegressor" in source or "class RareCategoryGrouper" in source):
        exec(source)

def build_preprocessing():
    categorical = Pipeline([("rare_grouping", RareCategoryGrouper(min_count=config["preprocessing"]["rare_min_count"])),
                            ("onehot", PlaceholderSafeOneHotEncoder())])
    columns = ColumnTransformer([("categorical", categorical, CATS), ("continuous", StandardScaler(), CONTS)],
                                remainder="drop", sparse_threshold=1.0)
    return Pipeline([("schema", FeatureSchemaValidator(expected_columns=FEATURES, continuous_columns=CONTS)),
                     ("preprocessing", columns)])

def build_model(alpha):
    m = config["model"]
    return LogTargetRegressor(Ridge(alpha=alpha, solver=m["solver"], tol=m["tol"], fit_intercept=m["fit_intercept"]))

In [4]:
ALPHAS = config["search_space"]["alpha"]
rows, oof = [], pd.Series(index=y.index, dtype=float)
start = perf_counter()
for k in range(5):
    fit_ids, held_ids = folds.index[folds.fold != k], folds.index[folds.fold == k]
    prep = build_preprocessing()
    Z_fit, Z_held = prep.fit_transform(X_train.loc[fit_ids]), prep.transform(X_train.loc[held_ids])
    for alpha in ALPHAS:
        pred = build_model(alpha).fit(Z_fit, y.loc[fit_ids]).predict(Z_held)
        errors = np.abs(pred - y.loc[held_ids].to_numpy())
        rows.append({"alpha": alpha, "fold": k, "claims": len(held_ids), "mae": errors.mean(), "absolute_error_sum": errors.sum()})
        if alpha == 100.0:
            oof.loc[held_ids] = pred
    print(f"Fold {k + 1}/5 done")
cv_seconds = perf_counter() - start
cv = pd.DataFrame(rows)

pooled = cv.groupby("alpha").absolute_error_sum.sum() / len(y)
tied = pooled.index[pooled <= pooled.min() + 1]
selected = float(max(tied))
print(f"CV time: {cv_seconds:.0f} s | tied: {list(tied)} | selected alpha: {selected:g}")

saved = pd.read_csv(OFFICIAL / "cv_results.csv").set_index(["alpha", "fold"]).sort_index()
gap = (cv.set_index(["alpha", "fold"]).sort_index().mae - saved.mae).abs().max()
print(f"Largest fold-MAE difference from the official run: {gap:.3f} loss units")
assert selected == official["selected_alpha"] == 100.0 and gap < 0.05

pd.DataFrame({"pooled CV MAE": pooled, "sd across folds": cv.groupby("alpha").mae.std()}).round(2)

Fold 1/5 done
Fold 2/5 done
Fold 3/5 done
Fold 4/5 done
Fold 5/5 done
CV time: 77 s | tied: [1.0, 10.0, 100.0] | selected alpha: 100
Largest fold-MAE difference from the official run: 0.000 loss units


,pooled CV MAE,sd across folds
alpha,,
1.0,1246.09,13.02
10.0,1245.72,13.18
100.0,1245.40,12.64
1000.0,1246.58,11.40
10000.0,1269.34,10.36
100000.0,1470.87,12.26


In [5]:
start = perf_counter()
final_prep = build_preprocessing()
Z_train = final_prep.fit_transform(X_train)
final = build_model(selected).fit(Z_train, y)
refit_seconds = perf_counter() - start

valid_pred = pd.Series(final.predict(final_prep.transform(X_valid)), index=X_valid.index)
saved_pred = pd.read_csv(OFFICIAL / "validation_predictions.csv").set_index("id").prediction.loc[valid_pred.index]
pred_gap = (valid_pred - saved_pred).abs().max()
print(f"Refit time: {refit_seconds:.0f} s | largest prediction difference: {pred_gap:.2f} loss units")
assert len(valid_pred) == 28248 and np.isfinite(valid_pred).all() and (valid_pred > 0).all()
assert ((valid_pred - saved_pred).abs() / saved_pred).max() < 1e-4

Refit time: 4 s | largest prediction difference: 0.00 loss units


In [6]:
train_fit_mae = np.abs(final.predict(Z_train) - y.to_numpy()).mean()
print(f"Training-fit MAE {train_fit_mae:,.2f} vs CV MAE {pooled[selected]:,.2f}")
print(f"Out-of-fold predictions: 5th-95th percentile {oof.quantile(0.05):,.0f} to {oof.quantile(0.95):,.0f};"
      f" training loss 95th percentile {y.quantile(0.95):,.0f}")

encoder = final_prep.named_steps["preprocessing"].named_transformers_["categorical"].named_steps["onehot"]
names = [(f, lvl) for f, lvls in zip(encoder.active_columns_, encoder.encoder_.categories_) for lvl in lvls]
names += [(c, "per 1 sd") for c in CONTS]
coef = pd.DataFrame(names, columns=["field", "level"])
coef["coefficient"] = final.regressor_.coef_
print("Model columns:", len(coef))

conts = coef[coef.level.eq("per 1 sd")].set_index("field").coefficient
print("\n% change per 1 sd (largest):")
print(((np.exp(conts) - 1) * 100).sort_values(key=abs, ascending=False).head(5).round(1))

cats = coef[coef.level.ne("per 1 sd")].groupby("field").coefficient
print("\nLargest multiplier between a field's highest and lowest label:")
print(np.exp(cats.max() - cats.min()).sort_values(ascending=False).head(5).round(2))

Training-fit MAE 1,240.42 vs CV MAE 1,245.40
Out-of-fold predictions: 5th-95th percentile 987 to 6,634; training loss 95th percentile 8,512
Model columns: 808

% change per 1 sd (largest):
field
cont2     5.7
cont7     4.4
cont1    -3.3
cont12    3.0
cont9     2.8
Name: coefficient, dtype: float64

Largest multiplier between a field's highest and lowest label:
field
cat57     1.49
cat100    1.48
cat77     1.42
cat80     1.41
cat113    1.41
Name: coefficient, dtype: float64


In [7]:
if not OUT.exists():
    OUT.mkdir(parents=True)
    cv.to_csv(OUT / "cv_results.csv", index=False)
    coef.to_csv(OUT / "coefficients.csv", index=False)
    (OUT / "reproduction_receipt.json").write_text(json.dumps({
        "run_id": OUT.name, "reproduces": official["run_id"], "owner": "Rachel Cheung", "issue": "#44",
        "created_by": "notebooks/gate3_approach1_glm.ipynb",
        "environment": {"python": platform.python_version(), **{p: version(p) for p in ["pandas", "numpy", "scipy", "scikit-learn"]}},
        "fold_seed": 42, "cv_fits": len(cv), "selected_alpha": selected,
        "pooled_cv_mae": {str(a): float(v) for a, v in pooled.items()},
        "training_fit_mae": float(train_fit_mae),
        "max_fold_mae_difference_vs_official": float(gap),
        "max_validation_prediction_difference_vs_official": float(pred_gap),
        "runtime_seconds": {"cv_30_fits": round(cv_seconds, 1), "final_refit": round(refit_seconds, 1)},
        "validation_targets_read": 0, "test_used": False,
    }, indent=2) + "\n")
print("Saved to", OUT.relative_to(ROOT))
print(f"\nDONE: alpha {selected:g}, CV MAE {pooled[selected]:,.2f}, official validation MAE {official['validation']['mae']:,.2f}")

Saved to artifacts/october/gate 3/runs/glm-repro-20261110-rachel

DONE: alpha 100, CV MAE 1,245.40, official validation MAE 1,250.45
